# Illustration Scrapping Studio ? Jupyter
Runs the complete app on this server. Your browser only controls and reviews it. No training integration.

Upload/clone the project first. Set `APP` below to its directory (the one containing `studio.py`). Keep `LIBRARY` on persistent storage. The notebook does not download an unknown repository or change your existing library location.

Node is needed only to build the frontend. If Node is absent, build with `npm ci` and `npm run build` in the laptop's `frontend` directory, then upload its entire `dist` directory to `APP/frontend/dist`. Setup can then use that build. Do not upload your laptop's `.venv`, credentials, or library as part of the source code.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil
APP = Path('/home/jovyan/Illustration-Scrapping-Studio').resolve()  # EDIT to the actual project directory
LIBRARY = Path('/home/jovyan/illustration-library').resolve()
PORT = 8000
assert (APP / 'studio.py').is_file(), 'Set APP to the directory containing studio.py'
assert sys.version_info >= (3, 12), 'Python 3.12+ required'
assert os.environ.get('JUPYTERHUB_SERVICE_PREFIX'), 'Run inside your authenticated JupyterHub session'
BASE_PATH = os.environ['JUPYTERHUB_SERVICE_PREFIX'].rstrip('/') + f'/proxy/{PORT}'
print('Persistent library:', LIBRARY)
print('Browser path:', BASE_PATH + '/')


In [ ]:
# Setup/update. Stop this app first before rerunning setup.
if 'studio_process' in globals() and studio_process.poll() is None:
    raise RuntimeError('Stop the app using the stop cell before updating')
command = [sys.executable, str(APP / 'studio.py'), 'setup']
if not shutil.which('npm'):
    assert (APP / 'frontend/dist/index.html').is_file(), 'Upload the built frontend/dist directory first, or install Node/npm'
    command.append('--skip-ui-build')
subprocess.run(command, cwd=APP, check=True)


In [ ]:
# Start once. A separate process continues when this browser disconnects.
# Server/container shutdown still stops it; restart explicitly after that.
import time, urllib.request, json
from IPython.display import display, HTML
from html import escape
if 'studio_process' in globals() and studio_process.poll() is None:
    raise RuntimeError('This notebook already started a running app')
LIBRARY.mkdir(parents=True, exist_ok=True)
launch_log = LIBRARY / 'launcher.log'
environment = dict(os.environ)
# Prevent inherited paths from silently targeting another library/database.
for key in ('ARTIST_DB_PATH', 'ARTIST_LIBRARY_PATH', 'ARTIST_LOG_PATH', 'ARTIST_PROVIDER_SETTINGS_PATH'):
    environment.pop(key, None)
with launch_log.open('ab') as output:
    studio_process = subprocess.Popen([str(APP / '.venv/bin/python'), str(APP / 'studio.py'), 'run', '--library', str(LIBRARY), '--database', str(LIBRARY / 'index.db'), '--port', str(PORT), '--base-path', BASE_PATH, '--no-browser'], cwd=APP, env=environment, stdout=output, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
for attempt in range(120):
    if studio_process.poll() is not None:
        raise RuntimeError(f'App exited. Read {launch_log}; check for an occupied port or locked library.')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=1) as response:
            health = json.load(response)
        if health['library_path'] != str(LIBRARY):
            raise RuntimeError('A different app already owns this port. Choose another PORT and rerun configuration.')
        time.sleep(1)
        if studio_process.poll() is not None:
            raise RuntimeError(f'App exited. Read {launch_log}')
        display(HTML(f'<a href="{escape(BASE_PATH + "/", quote=True)}" target="_blank" rel="noopener">Open Illustration Scrapping Studio</a>'))
        break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError(f'Startup not ready. Read {launch_log}; the process may still be starting.')


In [ ]:
# Stop only the process started by THIS notebook session. Run when needed.
# Existing job recovery applies when the backend starts again.
import signal
STOP_APP = False  # Set True only when you want to stop, then run this cell.
if not STOP_APP:
    print('App left running. Set STOP_APP=True to request shutdown.')
elif 'studio_process' in globals() and studio_process.poll() is None:
    studio_process.send_signal(signal.SIGTERM)
    print('Graceful shutdown requested. Active work may take time to finish; inspect launcher.log.')
else:
    print('No running process owned by this notebook session.')


## Reconnect and maintenance
Reopen the printed proxy link to reconnect. If the kernel restarts, do not blindly launch a second copy: the backend may still be running. Library locks and port checks reject duplicate starts. Use your terminal/process manager to stop the known app if the notebook has lost its process handle.

Provider credentials entered/uploaded through this UI are stored on the server. Use the authenticated Jupyter link; do not put this port behind a public anonymous tunnel. Back up privately with `studio.py backup` after stopping the backend; backups include credentials. See `docs/JUPYTER.md` for setup, updates and troubleshooting.
